In [0]:
# ============================================================
# CHESS LIVE ANALYTICS PROJECT
# SILVER LAYER CONFIGURATION
# ============================================================

BRONZE_TABLE = "grandmasters.bronze.tv_feed"

SILVER_GAMES = "grandmasters.silver.game_events"
SILVER_GAMES_CHECKPOINT_LOCATION = "/Volumes/grandmasters/silver/checkpoints/game_events"

spark.sql("""
          CREATE VOLUME IF NOT EXISTS grandmasters.silver.checkpoints
        COMMENT 'Checkpoint storage for streaming jobs';
          """)

In [0]:
# ============================================================
# READ BRONZE
# ============================================================

bronze_stream = spark.readStream.format("delta").table(BRONZE_TABLE)

In [0]:
# ============================================================
# EXTRACT GAME EVENTS
#
# Purpose:
# Parse 'featured' events from Bronze into structured columns.
# One record represents one chess game.
# ============================================================

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import *

featured_schema = StructType([
    StructField("t", StringType()),
    StructField("d", StructType([
        StructField("id", StringType()),
        StructField("orientation", StringType()),
        StructField("fen", StringType()),
        StructField("winner", StringType()),
        StructField("players", ArrayType(
            StructType([
                StructField("color", StringType()),
                StructField("rating", IntegerType()),
                StructField("seconds", IntegerType()),
                StructField("user", StructType([
                    StructField("name", StringType()),
                    StructField("id", StringType())
                ]))
            ])
        ))
    ]))
])

game_events_stream_df = (
    bronze_stream
    .filter(col("event_type") == "featured")
    .withColumn(
        "json",
        from_json(col("raw_event"), featured_schema)
    )
    .select(
        col("game_id"),
        col("ingested_at"),
        col("json.d.players")[0]["user"]["name"].alias("white_player"),
        col("json.d.players")[1]["user"]["name"].alias("black_player"),
        col("json.d.players")[0]["rating"].alias("white_rating"),
        col("json.d.players")[1]["rating"].alias("black_rating"),
        col("json.d.players")[0]["seconds"].alias("time_control_seconds"),
        col("json.d.winner").alias("winner"),
        col("json.d.fen").alias("starting_fen")
    )
)

query = (
    game_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_GAMES_CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(SILVER_GAMES)
)

query.awaitTermination()

print(f"Silver batch complete for game_events. Processed up to latest available bronze offset.")